# MaaS validation demo (pre-provisioned API key)

MaaS URL + **`API_KEY`** → list models → one completion → rate-limit loop (no OpenShift / key-creation steps).

**RHOAI:** **Demo quick swap** for `MAAS_BASE` + key between takes. README *OpenShift CLI and console* for `oc` on your laptop.

**Prereq:** Python 3.9+ (stdlib: `urllib`, `ssl`).

## Demo quick swap

Overrides for gateway URL and API key. Run this cell, then **Setup**.

| Variable | Effect |
|----------|--------|
| `DEMO_MAAS_BASE` | Non-empty → overrides `MAAS_BASE` env. |
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY` env. |

In [ ]:
DEMO_MAAS_BASE = ""
DEMO_API_KEY = ""

## Setup

Resolves `MAAS_BASE` / `API_KEY`, defines **`http_json`**, prints non-secret summary.

In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request
from typing import Any, Dict, Optional

_mb = globals().get("DEMO_MAAS_BASE", "")
if isinstance(_mb, str) and _mb.strip():
    MAAS_BASE = _mb.strip().rstrip("/")
else:
    MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE").strip().rstrip("/")

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")
MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"

if not API_KEY:
    raise SystemExit(
        "Set DEMO_API_KEY in the quick-swap cell or MAAS_API_KEY / API_KEY in the environment."
    )


def http_json(
    method: str,
    url: str,
    *,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
):
    """Minimal JSON HTTP helper (stdlib only)."""
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = None
    if data is not None:
        body = json.dumps(data).encode("utf-8")
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


print("MAAS_BASE :", MAAS_BASE)
print("MODELS_URL:", MODELS_URL)
print("VERIFY_TLS:", VERIFY_TLS)
print("API key set:", bool(API_KEY))

## 1. Model list (discovery)

`GET /maas-api/v1/models` with **`API_KEY`**. Parses first model → **`MODEL_NAME`**, **`MODEL_URL`**.

In [ ]:
_B = "\033[1m"
_R = "\033[0m"

_, models_body = http_json("GET", MODELS_URL, token=API_KEY)

data = models_body.get("data") or []
if not data:
    raise SystemExit("No models in response; deploy a model and check subscription binding.")

first = data[0]
MODEL_NAME = first.get("id") or first.get("name")
MODEL_URL = (first.get("url") or "").rstrip("/")
if not MODEL_NAME or not MODEL_URL:
    raise RuntimeError(f"Could not parse model id/url from: {first}")

print()
print("┌" + "─" * 58 + "┐")
print("│  " + _B + "MODEL_NAME" + _R + "  " + MODEL_NAME.ljust(42) + " │")
print("│  " + _B + "MODEL_URL " + _R + "  " + MODEL_URL[:42].ljust(42) + " │")
print("└" + "─" * 58 + "┘")
if len(MODEL_URL) > 42:
    print("   " + MODEL_URL[42:])
print()
print(_B + "Raw (truncated)" + _R)
print(json.dumps(models_body, indent=2)[:3500])

## 2. Single inference call

`POST {MODEL_URL}/v1/completions` with **`API_KEY`**. Prints usage tokens and completion text.

In [ ]:
_B = "\033[1m"
_R = "\033[0m"

COMPLETIONS_URL = f"{MODEL_URL}/v1/completions"
inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Hello from the notebook demo.",
    "max_tokens": 50,
}

status, completion = http_json("POST", COMPLETIONS_URL, token=API_KEY, data=inference_payload)

usage = completion.get("usage") or {}
choices = completion.get("choices") or []
choice0 = choices[0] if choices and isinstance(choices[0], dict) else {}
completion_text = choice0.get("text") or ""

print()
print(_B + "HTTP" + _R, status)
print()
print("┌" + "─" * 58 + "┐")
print("│  " + _B + "Tokens (usage)" + _R + " " * 29 + "│")
print("│    prompt_tokens      " + str(usage.get("prompt_tokens", "—")).ljust(31) + " │")
print("│    completion_tokens  " + str(usage.get("completion_tokens", "—")).ljust(31) + " │")
print("│    total_tokens       " + str(usage.get("total_tokens", "—")).ljust(31) + " │")
print("└" + "─" * 58 + "┘")
print()
print(_B + "Completion text" + _R)
print(completion_text if completion_text else "(empty)")
print()
print(_B + "Full JSON (truncated)" + _R)
print(json.dumps(completion, indent=2)[:4000])

## 3. Rate-limit probe

Repeated `POST` completions. Optional delay between requests. Stops after **`CONSECUTIVE_429_TO_STOP`** consecutive **429** responses, or **`MAX_REQUESTS`** total.

In [ ]:
import time

# --- tune here ---
SLEEP_BETWEEN_REQUESTS_SEC = 1.0  # 0 to disable pause between requests
CONSECUTIVE_429_TO_STOP = 3
MAX_REQUESTS = 64

inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Rate limit probe.",
    "max_tokens": 50,
}

ctx = ssl.create_default_context()
if not VERIFY_TLS:
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

consecutive_429 = 0
last_code = None
for i in range(1, MAX_REQUESTS + 1):
    body = json.dumps(inference_payload).encode("utf-8")
    req = urllib.request.Request(
        COMPLETIONS_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            last_code = resp.status
    except urllib.error.HTTPError as e:
        last_code = e.code

    print(f"{i:3d}  HTTP {last_code}", end="")
    if last_code == 429:
        consecutive_429 += 1
        print(f"  (429 streak {consecutive_429}/{CONSECUTIVE_429_TO_STOP})")
        if consecutive_429 >= CONSECUTIVE_429_TO_STOP:
            print(f"Stopping: {CONSECUTIVE_429_TO_STOP} consecutive 429 responses.")
            break
    else:
        consecutive_429 = 0
        print()

    if SLEEP_BETWEEN_REQUESTS_SEC > 0 and i < MAX_REQUESTS:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
else:
    if consecutive_429 < CONSECUTIVE_429_TO_STOP:
        print(f"Stopped: reached MAX_REQUESTS={MAX_REQUESTS} without {CONSECUTIVE_429_TO_STOP} consecutive 429s.")